## Calclate mutation rate for each Roulette bin for different ancestries

In [1]:
# --- make parent folder importable ---
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))  # points to Paper_SFS/

# --- usual imports ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy as sc
from scipy.integrate import quad
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from scipy.signal import find_peaks
from scipy.stats import gaussian_kde
from scipy.stats import gmean, linregress, norm, beta, uniform, lognorm
from scipy.integrate import simpson, trapezoid
from scipy.optimize import minimize
from scipy.special import logsumexp
from scipy.integrate import simpson
from scipy import stats
import glob
import os
import math
import dask
import dask.dataframe as dd
import re
import dask.bag as db
import csv
from tqdm import tqdm
from dask.diagnostics import ProgressBar
from concurrent.futures import ProcessPoolExecutor
import time
import joblib
from joblib import Parallel, delayed, parallel_backend
import multiprocessing
import pickle
import random
import json

# --- import your project modules (files in Paper_SFS/) ---
import Gen_SFS_with_s_demography
import plot_SFS

# (optional) pull specific functions
# from Gen_SFS_with_s import calc_SFS_k
# from plot_SFS import plot_sfs


In [2]:
def load_syn_anc(demography, data_root="../Data/syn_anc"):
    """
    Load synonymous ancestry-specific data for a given demography.

    Parameters
    ----------
    demography : str - e.g. 'NFE', 'AFR', 'EAS'
    data_root : str or Path - Path to Data/syn_anc directory

    Returns
    -------
    df : pandas.DataFrame
    allele_number : allele_number mode for that demography
    """
    data_root = Path(data_root)

    matches = sorted(data_root.glob(f"{demography}_*.txt.gz"))

    if len(matches) == 0:
        raise FileNotFoundError(f"No files found for demography '{demography}' in {data_root}")

    if len(matches) > 1:
        raise ValueError(f"Multiple files found for demography '{demography}':\n" + "\n".join(str(p.name) for p in matches))

    file_path = matches[0]

    # Extract the number e.g. "24590" from "NFE_24590.txt.gz"
    filename = os.path.basename(file_path)
    num_str = filename.split("_")[1].split(".")[0]   # split on "_" then remove ".txt.gz"
    allele_number = int(num_str)

    # Load the file
    df = pd.read_csv(file_path, sep="\t", compression="gzip")

    return df, allele_number


def load_demography_result(json_file, demography, epoch_no):
    with open(json_file, "r") as f:
        records = json.load(f)
    for rec in records:
        if rec["demography"] == demography and rec["epoch_no"] == epoch_no:
            return rec
    return {"params": []}


def log_likelihood_3_param(params, df_syn, SFS_low_mu, n, demography, params_dem, kmax = 5000):

    """
    Computes the negative log-likelihood for given values of mu
    - `params`: mean and var of mu and weight p
    - `df_syn`: DataFrame with allele_count and SFS counts (count)
    """
    mean_mu, var_mu, p = params

    _, _, _, SFS_neutral = Gen_SFS_with_s_demography.compute_SFS_gamma_mu_var(mu=mean_mu, var=var_mu, s=0, demography=demography, n=n, 
                                                                              params=params_dem, model="original", kmax=kmax)

    # Get k values as integer array
    k_indices = df_syn["allele_count"].astype(int).values

    # Get counts values as integer array
    counts = df_syn["count"].astype(int).values

    # Compute likelihood
    log_likelihood =  np.log((1-p)*SFS_neutral[k_indices]+p*SFS_low_mu[k_indices] + 1e-300) * counts

    logL = np.sum(log_likelihood)  # Sum log-likelihood across all loci

    return -logL  # Return negative log-likelihood for minimization



def optimize_for_MR_worker(args):

    i, MR_filtered, df_syn_filtered, SFS_low_mu, AN, demography, params_dem, kmax = args

    MR = MR_filtered[i]
    df_syn_analyze = df_syn_filtered[df_syn_filtered["MR"] == MR]

    # Initial params
    initial_params = [MR * 5e-8 * 4 * 14448, MR * 1e-15 * (4 * 14448) ** 2, 0.0001]

    bounds = [(2e-10 * 4 * 14448, 3e-6 * 4 * 14448), (1e-23 * (4 * 14448) ** 2, 1e-13 * (4 * 14448) ** 2), (0, 0.2)]

    result = minimize(
        log_likelihood_3_param,
        initial_params,
        args=(df_syn_analyze, SFS_low_mu, AN, demography, params_dem, kmax),
        bounds=bounds,
        method="Nelder-Mead",
        options={"disp": True, "fatol": 1e-6}
    )

    mean_mu = result.x[0]
    var_mu  = result.x[1]
    p_mu    = result.x[2]
    loglik  = -result.fun

    return i, MR, p_mu, mean_mu, var_mu, loglik


def mutation_rate_calc(demography, mu_low_mu = 2e-10, data_root = "../Data/syn_anc", json_file = "demography_results.json",
                       epoch = 3, kmax = 5000):

    # Load the QC'ed ancestry specific synonymous dataset- Contains Roulette rate (MR), allele count (allele_count), number of points corresponding 
    # to unique roulette rate and allele count (count)

    df, AN = load_syn_anc(demography, data_root = data_root)
    df_syn_filtered = df[df['allele_count']<=kmax]

    # Count of unique allele counts for each MR [SFS(k) - how many different k's are present for each MR]
    MR_counts = df_syn_filtered['MR'].value_counts().sort_index()
    # Filter MR values with more than 0 points
    MR_filtered = MR_counts[MR_counts > 0].index # Basically unique MR values which are non NA

    params_rec = load_demography_result(json_file, demography, epoch)

    _, _, _, SFS_low_mu = Gen_SFS_with_s_demography.compute_SFS(mu=mu_low_mu, s=0, demography=demography, n=AN, params=params_rec["params"], 
                                                                model="original", kmax=kmax)

    # Run in parallel and write results
    output_dir = Path("../Mutation_rate_estimation/param_mut_rate")
    output_dir.mkdir(exist_ok=True)

    if demography == "all" or demography == "NFE" or demography == "schraiber_et_al":
        output_file = output_dir / f"results_mu_var_p_{demography}.txt"
    else:
        output_file = output_dir / f"results_mu_var_p_{demography}_epoch_{epoch}.txt"
    
    with open(output_file, "w") as f:
        f.write("MR\tp_mu\tmean_mu\tvar_mu\tmax_loglikelihood\n")

        # Build argument list for worker
        # Note: range(1, len(MR_filtered)) to skip first roulette rate as in your original code
        tasks = [
            (i, MR_filtered, df_syn_filtered, SFS_low_mu, AN, demography, params_rec["params"], kmax)
            for i in range(1, len(MR_filtered))
        ]
    
        with ProcessPoolExecutor() as executor:
            for i, MR, p_mu, mean_mu, var_mu, loglik in executor.map(optimize_for_MR_worker, tasks):
                f.write(f"{MR}\t{p_mu}\t{mean_mu}\t{var_mu}\t{loglik}\n")

### Running the mutation rate estimation

In [3]:
demography = "EAS"

mutation_rate_calc(demography, mu_low_mu = 2e-10, data_root = "../Data/syn_anc", json_file = "demography_results.json", epoch = 3, kmax = 5000)

Optimization terminated successfully.
         Current function value: 68185.553138
         Iterations: 54
         Function evaluations: 96
Optimization terminated successfully.
         Current function value: 52437.374474
         Iterations: 61
         Function evaluations: 109
Optimization terminated successfully.
         Current function value: 110348.082566
         Iterations: 69
         Function evaluations: 127
Optimization terminated successfully.
         Current function value: 26209.828785
         Iterations: 62
         Function evaluations: 111
Optimization terminated successfully.
         Current function value: 32719.082361
         Iterations: 66
         Function evaluations: 123
Optimization terminated successfully.
         Current function value: 175566.767785
         Iterations: 103
         Function evaluations: 240
Optimization terminated successfully.
         Current function value: 185870.908639
         Iterations: 105
         Function evaluations:

/tmp/ipykernel_1554317/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(
/tmp/ipykernel_1554317/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


Optimization terminated successfully.
         Current function value: 1259.874852
         Iterations: 60
         Function evaluations: 111
Optimization terminated successfully.
         Current function value: 1343.020943
         Iterations: 47
         Function evaluations: 82
Optimization terminated successfully.
         Current function value: 10754.536215
         Iterations: 141
         Function evaluations: 355
Optimization terminated successfully.
         Current function value: 1134.334383
         Iterations: 55
         Function evaluations: 100
Optimization terminated successfully.
         Current function value: 1413.307382
         Iterations: 48
         Function evaluations: 89
Optimization terminated successfully.
         Current function value: 1558.131423
         Iterations: 53
         Function evaluations: 97
Optimization terminated successfully.
         Current function value: 1880.425264
         Iterations: 61
         Function evaluations: 104
Optimiz

/tmp/ipykernel_1554317/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(
/tmp/ipykernel_1554317/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(
/tmp/ipykernel_1554317/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


In [4]:
demography = "SAS"

mutation_rate_calc(demography, mu_low_mu = 2e-10, data_root = "../Data/syn_anc", json_file = "demography_results.json", epoch = 3, kmax = 5000)

Optimization terminated successfully.
         Current function value: 321533.338871
         Iterations: 61
         Function evaluations: 111
Optimization terminated successfully.
         Current function value: 90202.717349
         Iterations: 64
         Function evaluations: 114
Optimization terminated successfully.
         Current function value: 153714.131886
         Iterations: 64
         Function evaluations: 112
Optimization terminated successfully.
         Current function value: 119470.605373
         Iterations: 61
         Function evaluations: 112
Optimization terminated successfully.
         Current function value: 69552.364249
         Iterations: 64
         Function evaluations: 115
Optimization terminated successfully.
         Current function value: 193713.428864
         Iterations: 68
         Function evaluations: 120
Optimization terminated successfully.
         Current function value: 283654.771967
         Iterations: 65
         Function evaluations

/tmp/ipykernel_1554317/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(
/tmp/ipykernel_1554317/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


Optimization terminated successfully.
         Current function value: 4473.744532
         Iterations: 54
         Function evaluations: 97
Optimization terminated successfully.
         Current function value: 9088.297936
         Iterations: 48
         Function evaluations: 82
Optimization terminated successfully.
         Current function value: 19433.292086
         Iterations: 42
         Function evaluations: 75
Optimization terminated successfully.
         Current function value: 12398.480512
         Iterations: 49
         Function evaluations: 92
Optimization terminated successfully.
         Current function value: 12782.391095
         Iterations: 52
         Function evaluations: 89
Optimization terminated successfully.
         Current function value: 12890.118536
         Iterations: 48
         Function evaluations: 88
Optimization terminated successfully.
         Current function value: 19672.555338
         Iterations: 43
         Function evaluations: 83
Optimiza

/tmp/ipykernel_1554317/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


Optimization terminated successfully.
         Current function value: 22589.600819
         Iterations: 59
         Function evaluations: 110
Optimization terminated successfully.
         Current function value: 66155.020117
         Iterations: 54
         Function evaluations: 100
Optimization terminated successfully.
         Current function value: 59270.626103
         Iterations: 56
         Function evaluations: 105
Optimization terminated successfully.
         Current function value: 7006.071465
         Iterations: 63
         Function evaluations: 116
Optimization terminated successfully.
         Current function value: 1668.418467
         Iterations: 56
         Function evaluations: 102
Optimization terminated successfully.
         Current function value: 314.781514
         Iterations: 55
         Function evaluations: 99
Optimization terminated successfully.
         Current function value: 33204.216391
         Iterations: 158
         Function evaluations: 280
Opt

/tmp/ipykernel_1554317/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


In [5]:
demography = "AMR"

mutation_rate_calc(demography, mu_low_mu = 2e-10, data_root = "../Data/syn_anc", json_file = "demography_results.json", epoch = 3, kmax = 5000)

Optimization terminated successfully.
         Current function value: 109595.920530
         Iterations: 67
         Function evaluations: 127
Optimization terminated successfully.
         Current function value: 64310.950932
         Iterations: 80
         Function evaluations: 147
Optimization terminated successfully.
         Current function value: 169314.788914
         Iterations: 108
         Function evaluations: 263
Optimization terminated successfully.
         Current function value: 33116.186678
         Iterations: 60
         Function evaluations: 106
Optimization terminated successfully.
         Current function value: 40092.032125
         Iterations: 77
         Function evaluations: 135
Optimization terminated successfully.
         Current function value: 84569.674973
         Iterations: 114
         Function evaluations: 287
Optimization terminated successfully.
         Current function value: 169997.076077
         Iterations: 122
         Function evaluation

/tmp/ipykernel_1554317/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(
/tmp/ipykernel_1554317/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


Optimization terminated successfully.
         Current function value: 1499.143132
         Iterations: 58
         Function evaluations: 109
Optimization terminated successfully.
         Current function value: 27487.939707
         Iterations: 134
         Function evaluations: 359
Optimization terminated successfully.
         Current function value: 1505.468111
         Iterations: 55
         Function evaluations: 98
Optimization terminated successfully.
         Current function value: 1229.602816
         Iterations: 48
         Function evaluations: 86
Optimization terminated successfully.
         Current function value: 1464.045472
         Iterations: 50
         Function evaluations: 92
Optimization terminated successfully.
         Current function value: 18546.243071
         Iterations: 159
         Function evaluations: 382
Optimization terminated successfully.
         Current function value: 1447.729173
         Iterations: 54
         Function evaluations: 100
Optim

/tmp/ipykernel_1554317/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


Optimization terminated successfully.
         Current function value: 3226.251078
         Iterations: 189
         Function evaluations: 485
Optimization terminated successfully.
         Current function value: 34972.731279
         Iterations: 59
         Function evaluations: 112
Optimization terminated successfully.
         Current function value: 18278.179835
         Iterations: 68
         Function evaluations: 124


/tmp/ipykernel_1554317/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


Optimization terminated successfully.
         Current function value: 46005.638273
         Iterations: 57
         Function evaluations: 106
Optimization terminated successfully.
         Current function value: 1310.788576
         Iterations: 144
         Function evaluations: 348
Optimization terminated successfully.
         Current function value: 60161.545041
         Iterations: 55
         Function evaluations: 107
Optimization terminated successfully.
         Current function value: 17639.383181
         Iterations: 41
         Function evaluations: 79
Optimization terminated successfully.
         Current function value: 1465.330642
         Iterations: 44
         Function evaluations: 79
Optimization terminated successfully.
         Current function value: 62103.221545
         Iterations: 71
         Function evaluations: 126
Optimization terminated successfully.
         Current function value: 47589.122147
         Iterations: 66
         Function evaluations: 118
Op

/tmp/ipykernel_1554317/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(
/tmp/ipykernel_1554317/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


In [6]:
demography = "FIN"

mutation_rate_calc(demography, mu_low_mu = 2e-10, data_root = "../Data/syn_anc", json_file = "demography_results.json", epoch = 3, kmax = 5000)

Optimization terminated successfully.
         Current function value: 34189.350887
         Iterations: 40
         Function evaluations: 73
Optimization terminated successfully.
         Current function value: 42715.716805
         Iterations: 44
         Function evaluations: 82
Optimization terminated successfully.
         Current function value: 54027.383999
         Iterations: 50
         Function evaluations: 93
Optimization terminated successfully.
         Current function value: 90147.599602
         Iterations: 95
         Function evaluations: 223
Optimization terminated successfully.
         Current function value: 80944.534436
         Iterations: 101
         Function evaluations: 243
Optimization terminated successfully.
         Current function value: 89319.141533
         Iterations: 116
         Function evaluations: 278
Optimization terminated successfully.
         Current function value: 15192.478460
         Iterations: 118
         Function evaluations: 305

/tmp/ipykernel_1554317/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(
/tmp/ipykernel_1554317/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(
/tmp/ipykernel_1554317/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


Optimization terminated successfully.
         Current function value: 1093.772123
         Iterations: 28
         Function evaluations: 54
Optimization terminated successfully.
         Current function value: 4701.547851
         Iterations: 107
         Function evaluations: 302
Optimization terminated successfully.
         Current function value: 2490.190680
         Iterations: 104
         Function evaluations: 297
Optimization terminated successfully.
         Current function value: 3867.012382
         Iterations: 109
         Function evaluations: 308
Optimization terminated successfully.
         Current function value: 4721.894181
         Iterations: 119
         Function evaluations: 322
Optimization terminated successfully.
         Current function value: 1117.453787
         Iterations: 55
         Function evaluations: 105
Optimization terminated successfully.
         Current function value: 2249.676732
         Iterations: 78
         Function evaluations: 203
Opt

/tmp/ipykernel_1554317/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


Optimization terminated successfully.
         Current function value: 637.889136
         Iterations: 99
         Function evaluations: 264


/tmp/ipykernel_1554317/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


Optimization terminated successfully.
         Current function value: 1200.318461
         Iterations: 73
         Function evaluations: 198
Optimization terminated successfully.
         Current function value: 2614.126994
         Iterations: 47
         Function evaluations: 84
Optimization terminated successfully.
         Current function value: 489.296342
         Iterations: 92
         Function evaluations: 231
Optimization terminated successfully.
         Current function value: 3411.756201
         Iterations: 34
         Function evaluations: 64
Optimization terminated successfully.
         Current function value: 630.325287
         Iterations: 122
         Function evaluations: 335
Optimization terminated successfully.
         Current function value: 3731.607421
         Iterations: 50
         Function evaluations: 94
Optimization terminated successfully.
         Current function value: 4427.652538
         Iterations: 41
         Function evaluations: 70
Optimizatio

/tmp/ipykernel_1554317/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


Optimization terminated successfully.
         Current function value: 30670.010663
         Iterations: 146
         Function evaluations: 381
Optimization terminated successfully.
         Current function value: 784.634455
         Iterations: 136
         Function evaluations: 352
Optimization terminated successfully.
         Current function value: 122.655256
         Iterations: 157
         Function evaluations: 397


/tmp/ipykernel_1554317/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


In [3]:
demography = "AFR"

mutation_rate_calc(demography, mu_low_mu = 2e-10, data_root = "../Data/syn_anc", json_file = "demography_results.json", epoch = 3, kmax = 5000)

Optimization terminated successfully.
         Current function value: 68077.305765
         Iterations: 64
         Function evaluations: 111
Optimization terminated successfully.
         Current function value: 142866.831357
         Iterations: 62
         Function evaluations: 114
Optimization terminated successfully.
         Current function value: 174465.516381
         Iterations: 61
         Function evaluations: 113
Optimization terminated successfully.
         Current function value: 34317.741697
         Iterations: 53
         Function evaluations: 93
Optimization terminated successfully.
         Current function value: 180286.585122
         Iterations: 74
         Function evaluations: 130
Optimization terminated successfully.
         Current function value: 41787.210534
         Iterations: 64
         Function evaluations: 111
Optimization terminated successfully.
         Current function value: 28920.341882
         Iterations: 61
         Function evaluations: 1

/tmp/ipykernel_1363570/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


Optimization terminated successfully.
         Current function value: 19212.778821
         Iterations: 61
         Function evaluations: 117
Optimization terminated successfully.
         Current function value: 28155.443513
         Iterations: 55
         Function evaluations: 102
Optimization terminated successfully.
         Current function value: 3693.114374
         Iterations: 50
         Function evaluations: 89
Optimization terminated successfully.
         Current function value: 38182.097138
         Iterations: 75
         Function evaluations: 134
Optimization terminated successfully.
         Current function value: 3059.128719
         Iterations: 59
         Function evaluations: 112
Optimization terminated successfully.
         Current function value: 8342.827066
         Iterations: 51
         Function evaluations: 96
Optimization terminated successfully.
         Current function value: 63203.160840
         Iterations: 70
         Function evaluations: 129
Opti

/tmp/ipykernel_1363570/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(
/tmp/ipykernel_1363570/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


Optimization terminated successfully.
         Current function value: 44008.776595
         Iterations: 168
         Function evaluations: 411
Optimization terminated successfully.
         Current function value: 50751.586435
         Iterations: 197
         Function evaluations: 425
Optimization terminated successfully.
         Current function value: 67569.603622
         Iterations: 195
         Function evaluations: 462


/tmp/ipykernel_1363570/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(
/tmp/ipykernel_1363570/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


Optimization terminated successfully.
         Current function value: 64973.720951
         Iterations: 224
         Function evaluations: 486


/tmp/ipykernel_1363570/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


In [4]:
demography = "EAS"

mutation_rate_calc(demography, mu_low_mu = 2e-10, data_root = "../Data/syn_anc", json_file = "demography_results.json", epoch = 2, kmax = 5000)

Optimization terminated successfully.
         Current function value: 68270.862615
         Iterations: 60
         Function evaluations: 107
Optimization terminated successfully.
         Current function value: 110453.846258
         Iterations: 58
         Function evaluations: 102
Optimization terminated successfully.
         Current function value: 32748.501309
         Iterations: 67
         Function evaluations: 122
Optimization terminated successfully.
         Current function value: 52515.939827
         Iterations: 64
         Function evaluations: 121
Optimization terminated successfully.
         Current function value: 122856.815673
         Iterations: 115
         Function evaluations: 289
Optimization terminated successfully.
         Current function value: 158658.199325
         Iterations: 125
         Function evaluations: 329
Optimization terminated successfully.
         Current function value: 26238.817750
         Iterations: 49
         Function evaluations

/tmp/ipykernel_1363570/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(
/tmp/ipykernel_1363570/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


Optimization terminated successfully.
         Current function value: 22498.212841
         Iterations: 59
         Function evaluations: 105
Optimization terminated successfully.
         Current function value: 12559.793807
         Iterations: 67
         Function evaluations: 123
Optimization terminated successfully.
         Current function value: 4821.175197
         Iterations: 54
         Function evaluations: 99
Optimization terminated successfully.
         Current function value: 44568.575938
         Iterations: 47
         Function evaluations: 91
Optimization terminated successfully.
         Current function value: 49752.692474
         Iterations: 50
         Function evaluations: 94
Optimization terminated successfully.
         Current function value: 59286.206256
         Iterations: 55
         Function evaluations: 99
Optimization terminated successfully.
         Current function value: 50527.384388
         Iterations: 55
         Function evaluations: 102


/tmp/ipykernel_1363570/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


Optimization terminated successfully.
         Current function value: 39850.109949
         Iterations: 66
         Function evaluations: 122
Optimization terminated successfully.
         Current function value: 1087.271657
         Iterations: 37
         Function evaluations: 71
Optimization terminated successfully.
         Current function value: 22383.425622
         Iterations: 51
         Function evaluations: 97
Optimization terminated successfully.
         Current function value: 14783.425441
         Iterations: 56
         Function evaluations: 105
Optimization terminated successfully.
         Current function value: 8428.479793
         Iterations: 61
         Function evaluations: 114
Optimization terminated successfully.
         Current function value: 5063.937766
         Iterations: 53
         Function evaluations: 99
Optimization terminated successfully.
         Current function value: 14832.451372
         Iterations: 58
         Function evaluations: 110
Optim

/tmp/ipykernel_1363570/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(
/tmp/ipykernel_1363570/3654586471.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


In [ ]:
demography = "SAS"

mutation_rate_calc(demography, mu_low_mu = 2e-10, data_root = "../Data/syn_anc", json_file = "demography_results.json", epoch = 2, kmax = 5000)

Optimization terminated successfully.
         Current function value: 90384.559492
         Iterations: 60
         Function evaluations: 112
Optimization terminated successfully.
         Current function value: 119675.074653
         Iterations: 70
         Function evaluations: 125
Optimization terminated successfully.
         Current function value: 69663.520127
         Iterations: 69
         Function evaluations: 130


In [ ]:
demography = "AMR"

mutation_rate_calc(demography, mu_low_mu = 2e-10, data_root = "../Data/syn_anc", json_file = "demography_results.json", epoch = 2, kmax = 5000)

In [ ]:
demography = "FIN"

mutation_rate_calc(demography, mu_low_mu = 2e-10, data_root = "../Data/syn_anc", json_file = "demography_results.json", epoch = 2, kmax = 5000)

In [ ]:
demography = "AFR"

mutation_rate_calc(demography, mu_low_mu = 2e-10, data_root = "../Data/syn_anc", json_file = "demography_results.json", epoch = 2, kmax = 5000)

In [7]:
demography = "NFE"

mutation_rate_calc(demography, mu_low_mu = 2e-10, data_root = "../Data/syn_anc", json_file = "demography_results.json", epoch = 2, kmax = 5000)

Optimization terminated successfully.
         Current function value: 810046.677756
         Iterations: 109
         Function evaluations: 255
Optimization terminated successfully.
         Current function value: 842807.318386
         Iterations: 123
         Function evaluations: 275
Optimization terminated successfully.
         Current function value: 827840.316209
         Iterations: 115
         Function evaluations: 268
Optimization terminated successfully.
         Current function value: 671669.635720
         Iterations: 120
         Function evaluations: 297
Optimization terminated successfully.
         Current function value: 413030.983328
         Iterations: 124
         Function evaluations: 320
Optimization terminated successfully.
         Current function value: 785543.015529
         Iterations: 115
         Function evaluations: 316
Optimization terminated successfully.
         Current function value: 658984.217696
         Iterations: 126
         Function ev

/tmp/ipykernel_1757974/2288477654.py:87: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


Optimization terminated successfully.
         Current function value: 36728.288158
         Iterations: 157
         Function evaluations: 276
Optimization terminated successfully.
         Current function value: 90012.668560
         Iterations: 136
         Function evaluations: 244
Optimization terminated successfully.
         Current function value: 127106.225968
         Iterations: 126
         Function evaluations: 228
Optimization terminated successfully.
         Current function value: 59780.835475
         Iterations: 179
         Function evaluations: 319
Optimization terminated successfully.
         Current function value: 113514.372875
         Iterations: 148
         Function evaluations: 267
Optimization terminated successfully.
         Current function value: 146436.022187
         Iterations: 123
         Function evaluations: 213
Optimization terminated successfully.
         Current function value: 2535.492724
         Iterations: 89
         Function evaluati

In [3]:
demography = "all"

mutation_rate_calc(demography, mu_low_mu = 2e-10, data_root = "../Data/syn_anc", json_file = "demography_results.json", epoch = 2, kmax = 5000)

Optimization terminated successfully.
         Current function value: 948074.309779
         Iterations: 103
         Function evaluations: 261
Optimization terminated successfully.
         Current function value: 1329497.500112
         Iterations: 110
         Function evaluations: 281
Optimization terminated successfully.
         Current function value: 557926.046338
         Iterations: 116
         Function evaluations: 290
Optimization terminated successfully.
         Current function value: 246406.976256
         Iterations: 116
         Function evaluations: 321
Optimization terminated successfully.
         Current function value: 1071685.100381
         Iterations: 126
         Function evaluations: 325
Optimization terminated successfully.
         Current function value: 1076667.217735
         Iterations: 128
         Function evaluations: 331
Optimization terminated successfully.
         Current function value: 1255452.122547
         Iterations: 129
         Functio

/tmp/ipykernel_803935/1725332265.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(
/tmp/ipykernel_803935/1725332265.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


Optimization terminated successfully.
         Current function value: 192120.772479
         Iterations: 146
         Function evaluations: 362
Optimization terminated successfully.
         Current function value: 128812.101145
         Iterations: 121
         Function evaluations: 335
Optimization terminated successfully.
         Current function value: 16485.596187
         Iterations: 138
         Function evaluations: 245
Optimization terminated successfully.
         Current function value: 155423.869454
         Iterations: 140
         Function evaluations: 354
Optimization terminated successfully.
         Current function value: 7072.415629
         Iterations: 49
         Function evaluations: 93
Optimization terminated successfully.
         Current function value: 6473.030483
         Iterations: 56
         Function evaluations: 103
Optimization terminated successfully.
         Current function value: 9428.163257
         Iterations: 95
         Function evaluations: 

/tmp/ipykernel_803935/1725332265.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


Optimization terminated successfully.
         Current function value: 19981.179360
         Iterations: 61
         Function evaluations: 113
Optimization terminated successfully.
         Current function value: 27498.228936
         Iterations: 66
         Function evaluations: 120
Optimization terminated successfully.
         Current function value: 10106.979664
         Iterations: 91
         Function evaluations: 163


/tmp/ipykernel_803935/1725332265.py:86: RuntimeWarning: Maximum number of function evaluations has been exceeded.
  result = minimize(


Optimization terminated successfully.
         Current function value: 26689.407226
         Iterations: 81
         Function evaluations: 146
Optimization terminated successfully.
         Current function value: 27590.056140
         Iterations: 136
         Function evaluations: 237
Optimization terminated successfully.
         Current function value: 42974.219231
         Iterations: 135
         Function evaluations: 238
Optimization terminated successfully.
         Current function value: 40470.670425
         Iterations: 145
         Function evaluations: 256
Optimization terminated successfully.
         Current function value: 92568.350012
         Iterations: 132
         Function evaluations: 241
Optimization terminated successfully.
         Current function value: 103486.610795
         Iterations: 138
         Function evaluations: 243
Optimization terminated successfully.
         Current function value: 69078.676203
         Iterations: 148
         Function evaluatio